### TypedDict 格式的使用

#### 1. 理解

In [1]:
from typing_extensions import TypedDict

class MovieDict(TypedDict):
    title: str
    year: int
    director: str
    rating: float

movie: MovieDict = {
    "title1": "盗梦空间",
    "year": 2010,
    "director": "颗粒",
    "rating": 8.8
}
print(movie)

{'title1': '盗梦空间', 'year': 2010, 'director': '颗粒', 'rating': 8.8}


#### 2. 基本使用

In [3]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [4]:
from typing_extensions import Annotated

class MovieTypedDict(TypedDict):
    """电影的信息"""
    title: Annotated[str, "电影的名称"]
    year: Annotated[int, "电影的上映时间, 四位数"]
    director: Annotated[str, "电影的导演"]
    rating: Annotated[float, "电影的评分, 满分是 10 分, 可以包含一位小数"]

structured_model = model.with_structured_output(MovieTypedDict)
response = structured_model.invoke("给我介绍一下电影 星级穿越")
print(response)
print(type(response))

{'title': '《星际穿越》(Interstellar) 影片介绍', 'rating': 9.4}
<class 'dict'>


举例2: 嵌套结构的使用

In [5]:
from typing_extensions import Annotated
from typing import List

class Actor(TypedDict):
    name: Annotated[str, "演员名称"]
    role: Annotated[str, "演员角色"]

class MovieTypedDict(TypedDict):
    """电影的信息"""
    title: Annotated[str, "电影的名称"]
    year: Annotated[int, "电影的上映时间, 四位数"]
    director: Annotated[str, "电影的导演"]
    rating: Annotated[float, "电影的评分, 满分是 10 分, 可以包含一位小数"]
    cast: Annotated[List[Actor], "演员的列表"]


structured_model = model.with_structured_output(MovieTypedDict)
response = structured_model.invoke("给我介绍一下电影 星级穿越")
print(response)
print(type(response))

{'title': '《星际穿越》 介绍', 'director': '克里斯托弗·诺兰', 'cast': [{'name': '马修·麦康纳', 'role': '库珀'}, {'name': '安妮·海瑟薇', 'role': '艾米莉亚·布兰德'}, {'name': '杰西卡·查斯坦', 'role': '墨菲'}, {'name': '迈克尔·凯恩', 'role': '布兰德教授'}]}
<class 'dict'>


举例3: ... 的使用

说明: 下述代码的 ... 是 Python 的字面量, 等价于 Ellipsis, 可以理解为占位符, 下游框架 (如 LangChain) 可以对 ... 作定制化处理, 如 LangChain 中 Annotated 的 ... 表示当前字段是必须存在的, 不可省略, 用来指示模型的输出

In [6]:
class MovieTypedDict(TypedDict):
    """电影的信息"""
    title: Annotated[str, ..., "电影的名称"]
    year: Annotated[int, ..., "电影的上映时间, 四位数"]
    director: Annotated[str, ..., "电影的导演"]
    rating: Annotated[float, ..., "电影的评分, 满分是 10 分, 可以包含一位小数"]


structured_model = model.with_structured_output(MovieTypedDict)
response = structured_model.invoke("根据这句话抽取盗梦空间的信息, 不包含的信息可以留空: 盗梦空间在 2010 年上映, 导演是克里斯托弗")
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗', 'rating': 0}
<class 'dict'>


#### 2.3: JSON Schema

这种方式需要按照 JSON Schema 规范拼接 JSON 字符串, 比较繁琐, 并且缺少校验机制, 不推荐

举例1: 返回简单结构

method: 结构化输出的方式, 但是否可用, 依赖于模型供应商及 LangChain 适配器的具体实现, 比如, DeepSeek 模型服务不支持 json_schema 模式

json_schema: 使用模型供应商提供的专用结构化输出功能

以上代码中定义 json_schema 的时候指定的 title, description, type, properties, required 是遵循 JSON Schema 规范的标准关键字, 是固定写法, 几个关键字解释如
title: 为整个 Schema 或特定属性提供一个人类可读的标题, 不能是中文, 用于提高可读性

description: 提供更详细的文字描述, 说明 Schema 或属性的用途等, 和 title 一样, 皆在帮助理解

type: 定义当前数据节点必须是什么数据类型, 常见类型有 string, number, integer, boolean, object, array, null. object 即是 json 对象

properties: 用于定义 JSON 对象(object) 中可以包含哪些属性键, 以及每个属性对应的值类型和说明

required: 当 type 为 object 时使用, 是一个数组, 列出了对象中必须存在的属性名

In [7]:
json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

structured_model = model.with_structured_output(json_schema, method="json_schema")
response = structured_model.invoke("给我盗梦空间信息")
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.4}
<class 'dict'>
